# E3-08 - SFT con LoRA/QLoRA en Google Colab

Entrena `Qwen/Qwen2.5-1.5B-Instruct` para que reciba los **puntajes brutos** de
CHASIDE, Felder-Silverman y Personalidad y devuelva un **perfil integrado de 4
secciones**, y luego exporta el modelo a **GGUF `Q4_K_M`** para correr en CPU.

**Guia completa:** `docs/entregas/entrega-3-interpretacion-psicometrica/guia-entrenamiento-colab.md`

## Reglas

1. **Solo datos sinteticos.** Ningun perfil real, nombre, correo o codigo en Colab.
2. Ejecutar las celdas **en orden**. Si Colab pide reiniciar el entorno tras instalar, reiniciar y volver a la celda 2.
3. Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU`.

Tiempo estimado con 150 registros: 15 a 25 minutos de entrenamiento.


In [ ]:
# Celda 2 - Verificar GPU y version de PyTorch
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(props.total_memory / 1e9, 1))
    assert props.total_memory / 1e9 >= 12, "Se necesita al menos 12 GB de VRAM (T4 o superior)"
else:
    print("SIN GPU: activa Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU")


In [ ]:
# Celda 3 - Configuracion del experimento
from pathlib import Path

REPO_URL = "https://github.com/ISCOUTB/savio-ai-recommender.git"
MODO = "clonar"                      # "clonar" el repo o "subir" el JSONL a mano
MODELO_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
CORPUS_PREFERIDO = "datos/corpus/corpus.jsonl"
CORPUS_EJEMPLO = "datos/corpus/ejemplo_mini.jsonl"
DIR_SALIDA = Path("/content/drive/MyDrive/e3-08")
EPOCAS = 3                           # usar 1 con el ejemplo mini
MAX_LEN = 2048
SEMILLA = 42

print("Modelo base:", MODELO_BASE)


In [ ]:
# Celda 4 - Instalar dependencias (Colab ya trae PyTorch con CUDA)
import subprocess
import sys

PAQUETES = [
    "transformers>=4.46,<6",
    "peft>=0.13,<1",
    "datasets>=2.20,<5",
    "accelerate>=0.34,<2",
    "bitsandbytes>=0.43",
    "sentencepiece>=0.2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES], check=True)

import transformers, peft, bitsandbytes
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)


In [ ]:
# Celda 5 - Montar Drive, obtener el repo y localizar el corpus
import os
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_SALIDA.mkdir(parents=True, exist_ok=True)
    print("Drive montado en:", DIR_SALIDA)
except Exception as exc:
    DIR_SALIDA = Path("/content/e3-08")
    DIR_SALIDA.mkdir(parents=True, exist_ok=True)
    print("Sin Drive, guardando en", DIR_SALIDA, "->", exc)

if MODO == "clonar":
    repo = Path("savio-ai-recommender")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(repo)
    print("Directorio de trabajo:", Path.cwd())
else:
    from google.colab import files
    subidos = files.upload()
    Path("datos/corpus").mkdir(parents=True, exist_ok=True)
    for nombre in subidos:
        Path("datos/corpus", nombre).write_bytes(subidos[nombre])
    print("Archivos subidos:", list(subidos))

RUTA_CORPUS = Path(CORPUS_PREFERIDO)
if not RUTA_CORPUS.exists():
    RUTA_CORPUS = Path(CORPUS_EJEMPLO)
    EPOCAS = 1
    print("AVISO: no hay corpus.jsonl; se usa el ejemplo mini y 1 epoca.")

print("Corpus en uso:", RUTA_CORPUS)


In [ ]:
# Celda 6 - Gate de privacidad (obligatorio antes de entrenar)
import subprocess
import sys

validador = Path("docs/especificaciones/validar-corpus-anonimizado.py")
if validador.exists():
    corrida = subprocess.run(
        [sys.executable, str(validador), "--entrada", str(RUTA_CORPUS), "--manifiesto", str(DIR_SALIDA / "manifiesto.json")]
    )
    assert corrida.returncode == 0, "El corpus NO paso el gate de privacidad: no entrenar ni subir nada"
    print("Gate de privacidad: OK")
else:
    print("AVISO: validador no disponible (MODO='subir'). Ejecutalo en local antes de entrenar.")


In [ ]:
# Celda 7 - Cargar corpus y partir 80/20 sin fuga
import hashlib
import json
from pathlib import Path

registros = [
    json.loads(linea)
    for linea in Path(RUTA_CORPUS).read_text(encoding="utf-8").splitlines()
    if linea.strip()
]
assert registros, "El corpus esta vacio"

def es_evaluacion(registro):
    return int(hashlib.sha256(registro["id"].encode()).hexdigest(), 16) % 5 == 0

entrenamiento = [r for r in registros if not es_evaluacion(r)]
evaluacion = [r for r in registros if es_evaluacion(r)]

print("Total:", len(registros))
print("Entrenamiento:", len(entrenamiento))
print("Evaluacion:", len(evaluacion))
assert entrenamiento, "Sin registros de entrenamiento"

with open(DIR_SALIDA / "corpus_manifiesto_sha256.txt", "w", encoding="utf-8") as f:
    f.write(hashlib.sha256(Path(RUTA_CORPUS).read_bytes()).hexdigest() + "\n")


In [ ]:
# Celda 8 - Tokenizer y modelo base en 4-bit + adaptadores LoRA
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # la T4 no soporta bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=bnb,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
modelo = prepare_model_for_kbit_training(modelo, use_gradient_checkpointing=True)
modelo.config.use_cache = False

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
modelo = get_peft_model(modelo, lora)
modelo.print_trainable_parameters()


In [ ]:
# Celda 9 - Contrato de la tarea y tokenizacion con mascara de loss
SYSTEM_E3 = (
    "Eres un asistente de orientacion psicopedagogica. Recibes los puntajes brutos "
    "de CHASIDE, Felder-Silverman y el test de personalidad, sin transformarlos. "
    "Respondes siempre en cuatro secciones: 1. Estilos de aprendizaje, 2. Rasgos de "
    "personalidad, 3. Perfil vocacional, 4. Sintesis integrada. Citas los puntajes "
    "recibidos, no inventas escalas, no emites diagnosticos ni etiquetas de tipo de "
    "personalidad, y cierras con el aviso de que el perfil es un apoyo al diagnostico "
    "inicial y no reemplaza al profesional."
)

CAMPOS_PERFIL = ("chaside", "ils", "personalidad")

def perfil_legible(registro):
    base = registro.get("input") if isinstance(registro.get("input"), dict) else registro
    lineas = []
    for campo in CAMPOS_PERFIL:
        if campo in base:
            lineas.append(f"{campo.upper()}: " + json.dumps(base[campo], ensure_ascii=True))
    return "Puntajes brutos:\n" + "\n".join(lineas)

def mensajes(registro):
    return [
        {"role": "system", "content": SYSTEM_E3},
        {"role": "user", "content": perfil_legible(registro)},
        {"role": "assistant", "content": registro["output"]},
    ]

def tokenizar(registro):
    turnos = mensajes(registro)
    texto_prompt = tokenizer.apply_chat_template(
        turnos[:-1], tokenize=False, add_generation_prompt=True
    )
    texto_total = texto_prompt + turnos[-1]["content"] + tokenizer.eos_token
    ids_prompt = tokenizer(texto_prompt, add_special_tokens=False)["input_ids"]
    codificado = tokenizer(
        texto_total, truncation=True, max_length=MAX_LEN, add_special_tokens=False
    )
    etiquetas = list(codificado["input_ids"])
    corte = min(len(ids_prompt), len(etiquetas))
    etiquetas[:corte] = [-100] * corte  # solo se aprende el informe, no el prompt
    codificado["labels"] = etiquetas
    return codificado

ejemplo = tokenizar(entrenamiento[0])
enmascarados = sum(1 for v in ejemplo["labels"] if v == -100)
print("Tokens totales:", len(ejemplo["input_ids"]), "| enmascarados:", enmascarados)
assert enmascarados > 0 and enmascarados < len(ejemplo["labels"]), "La mascara de loss no se aplico"


In [ ]:
# Celda 10 - Dataset y collator con padding
import torch
from torch.utils.data import Dataset

class DatasetSFT(Dataset):
    def __init__(self, registros):
        self.ejemplos = [tokenizar(r) for r in registros]

    def __len__(self):
        return len(self.ejemplos)

    def __getitem__(self, indice):
        return self.ejemplos[indice]

class CollatorSFT:
    def __call__(self, lote):
        largo = max(len(x["input_ids"]) for x in lote)
        pad = tokenizer.pad_token_id
        salida = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in lote:
            relleno = largo - len(x["input_ids"])
            salida["input_ids"].append(x["input_ids"] + [pad] * relleno)
            salida["attention_mask"].append([1] * len(x["input_ids"]) + [0] * relleno)
            salida["labels"].append(x["labels"] + [-100] * relleno)
        return {clave: torch.tensor(valor) for clave, valor in salida.items()}

ds_entrenamiento = DatasetSFT(entrenamiento)
ds_evaluacion = DatasetSFT(evaluacion) if evaluacion else None
print("Ejemplos listos:", len(ds_entrenamiento), "| evaluacion:", len(ds_evaluacion or []))


In [ ]:
# Celda 11 - Entrenamiento
from transformers import Trainer, TrainingArguments

argumentos = TrainingArguments(
    output_dir=str(DIR_SALIDA / "checkpoints"),
    num_train_epochs=EPOCAS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=1,
    eval_strategy="epoch" if ds_evaluacion else "no",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEMILLA,
    remove_unused_columns=False,
)

entrenador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=ds_entrenamiento,
    eval_dataset=ds_evaluacion,
    data_collator=CollatorSFT(),
)

resultado = entrenador.train()
print("Loss final de entrenamiento:", round(resultado.training_loss, 4))
for entrada in entrenador.state.log_history:
    if "eval_loss" in entrada:
        print("eval_loss:", round(entrada["eval_loss"], 4), "| epoca:", entrada.get("epoch"))


In [ ]:
# Celda 12 - Generar sobre evaluacion y pasar el guardrail etico
import sys
from pathlib import Path

sys.path.insert(0, str(Path("poc_slm").resolve()))
try:
    from guardrail_etico import agregar_aviso, validar_salida_etica  # noqa: E402
except Exception:
    AVISO = (
        "Este perfil es un apoyo para el diagnostico inicial y fue generado con "
        "asistencia de IA. No reemplaza al profesional ni constituye un diagnostico "
        "psicologico."
    )

    def agregar_aviso(texto):
        return texto if AVISO.lower() in texto.lower() else texto.rstrip() + "\n\n---\n" + AVISO

    def validar_salida_etica(texto):
        return [] if AVISO.lower() in (texto or "").lower() else ["falta el aviso no clinico"]

modelo.eval()

def generar(registro, max_new_tokens=400):
    turnos = mensajes(registro)[:-1]
    entrada = tokenizer.apply_chat_template(
        turnos, add_generation_prompt=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(
            input_ids=entrada,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True).strip()

muestra = (evaluacion or entrenamiento)[:3]
for registro in muestra:
    texto = agregar_aviso(generar(registro))
    print("=" * 80)
    print("CASO:", registro["id"])
    print(texto)
    print("GUARDRAIL:", validar_salida_etica(texto) or "sin hallazgos")


In [ ]:
# Celda 13 - Unir el adaptador y guardar el modelo completo
from pathlib import Path

RUTA_MERGED = DIR_SALIDA / "merged"
modelo_merge = modelo.merge_and_unload()
modelo_merge.save_pretrained(RUTA_MERGED, safe_serialization=True)
tokenizer.save_pretrained(RUTA_MERGED)

tamano = sum(p.stat().st_size for p in RUTA_MERGED.rglob("*") if p.is_file())
print("Modelo guardado en:", RUTA_MERGED)
print("Tamano en disco GB:", round(tamano / 1e9, 2))


In [ ]:
# Celda 14 - Convertir a GGUF y cuantizar a Q4_K_M
!git clone --depth 1 https://github.com/ggml-org/llama.cpp || true
!cd llama.cpp && cmake -B build -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF && cmake --build build --config Release -j 4
!python llama.cpp/convert_hf_to_gguf.py {RUTA_MERGED} --outfile {DIR_SALIDA}/modelo-f16.gguf --outtype f16
!./llama.cpp/build/bin/llama-quantize {DIR_SALIDA}/modelo-f16.gguf {DIR_SALIDA}/modelo-q4_k_m.gguf Q4_K_M
!ls -lh {DIR_SALIDA}/*.gguf


In [ ]:
# Celda 15 - Descargar el GGUF (y publicar en Hugging Face si hay token)
from pathlib import Path

ruta_gguf = DIR_SALIDA / "modelo-q4_k_m.gguf"
print("GGUF:", ruta_gguf, ruta_gguf.exists())

try:
    from google.colab import files
    files.download(str(ruta_gguf))
except Exception as exc:
    print("Descarga manual desde el panel de archivos:", exc)

try:
    from google.colab import userdata
    from huggingface_hub import HfApi, login

    login(token=userdata.get("HF_TOKEN"))
    api = HfApi()
    repo = "perfil-ia-e3-qwen2.5-1.5b-qlora"
    api.create_repo(repo_id=repo, exist_ok=True)
    api.upload_folder(folder_path=str(RUTA_MERGED), repo_id=repo)
    api.upload_file(path_or_fileobj=str(ruta_gguf), path_in_repo="modelo-q4_k_m.gguf", repo_id=repo)
    print("Publicado en https://huggingface.co/" + repo)
except Exception as exc:
    print("Sin publicacion en Hugging Face:", exc)


## Cierre: evidencia obligatoria de E3-08

Registrar en `experimentos/bitacora.md`:

| Dato | Donde sale |
|---|---|
| Version de librerias | Salida de la celda 4 |
| Hash del corpus | `DIR_SALIDA/corpus_manifiesto_sha256.txt` |
| Numero de registros y split | Salida de la celda 7 |
| Hiperparametros | Celda 11 (`EPOCAS`, `MAX_LEN`, batch, acumulacion) |
| Loss final de entrenamiento y evaluacion | Salida de la celda 11 |
| Tamano del GGUF | Salida de la celda 14 |
| Latencia en CPU | Medicion local con Ollama (seccion 11 de la guia) |

Falta despues: crear el modelo en Ollama con el `Modelfile`, medir latencia por
perfil y adjuntar una salida de ejemplo con las 4 secciones y el aviso no clinico.
